<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/paper_summary/00_prepare_SLCP_samples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 00 — Prepare the paired SLCP simulation banks

This stage is the only training-data simulator entry point.  It generates one
master prior-predictive SLCP bank and materializes deterministic nested budget
views.  A fixed master-row assignment supplies the common train/validation
split to every matched flow and classifier.  It also creates the separate,
fixed two-row shape-inference bank, two-row consistency/ActNorm pilot, and
300-row validation bank required by the exact upstream JANA process; its
reported simulator cost is therefore N training calls plus 304.  A fifth,
separately seeded audit bank is never exposed to model or proposal selection.

The manifest verifies row counts, nested splits, non-overlap of training and
validation indices, distinct five-bank provenance, and SHA256 array
fingerprints.  All later notebooks load these artifacts and must fail loudly
if any identity check changes.


In [2]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization in this process.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("PAPER_SUMMARY_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

def installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP"
        )
    else:
        default_artifact_root = Path("/content/paper_summary_SLCP_artifacts")

    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, repository, env=clone_env,
        )
    else:
        run("git", "-C", repository, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", repository, "fetch", "origin", BRANCH)
        run("git", "-C", repository, "checkout", BRANCH)
        run("git", "-C", repository, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", repository, "sparse-checkout", "set", "src",
        "workshops/ml4hep_tifr_colab/paper_summary",
    )
    SOURCE_DIR = repository / "workshops" / "ml4hep_tifr_colab" / "paper_summary"

    # Colab already provides the numerical/ML stack used by these notebooks.
    # Install only the two missing modern-runtime packages normally.  In
    # particular, do not let sbibm pull its historical algorithm dependency
    # tree into the current Colab Python environment (currently Python 3.13).
    modern_requirements = []
    if installed_version("nflows") != "0.14":
        modern_requirements.append("nflows==0.14")
    if importlib.util.find_spec("pyro") is None:
        modern_requirements.append("pyro-ppl")
    if modern_requirements:
        run(sys.executable, "-m", "pip", "install", "-q", *modern_requirements)
    if installed_version("sbibm") != "1.1.0":
        # This is the same Python-3.13-safe installation used by Exercises 9
        # and 10: the SLCP task/metrics need sbibm itself, nflows, and Pyro,
        # but not sbibm's old pinned SBI/algorithm environment.
        run(
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "sbibm==1.1.0",
        )
else:
    candidates = (
        Path.cwd(),
        Path.cwd() / "paper_summary",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab" / "paper_summary",
    )
    SOURCE_DIR = next(
        (candidate.resolve() for candidate in candidates if (candidate / "config.py").is_file()),
        None,
    )
    if SOURCE_DIR is None:
        raise FileNotFoundError("Cannot locate the paper_summary source directory")
    default_artifact_root = SOURCE_DIR / "artifacts"

source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(SOURCE_DIR)

ARTIFACT_ROOT = Path(
    os.environ.get("PAPER_SUMMARY_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Paper-summary source:", SOURCE_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)


Paper-summary source: /content/nsbi-lhc-toolkit/workshops/ml4hep_tifr_colab/paper_summary
Persistent artifact root: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP


In [3]:
from config import (
    DEFAULT_ML_SEEDS,
    PAPER_BUDGETS,
    SMOKE_BUDGETS,
    SMOKE_ML_SEEDS,
    campaign_config,
    campaign_signature,
)

PROFILE = os.environ.get("PAPER_SUMMARY_PROFILE", "PAPER").upper()
CAMPAIGN_BUDGETS = list(PAPER_BUDGETS if PROFILE == "PAPER" else SMOKE_BUDGETS)
CAMPAIGN_ML_SEEDS = list(DEFAULT_ML_SEEDS if PROFILE == "PAPER" else SMOKE_ML_SEEDS)

def execution_subset(environment_name, configured):
    raw = os.environ.get(environment_name, "").strip()
    values = list(configured) if not raw else [int(value) for value in raw.split(",")]
    unknown = set(values) - set(configured)
    if not values or unknown:
        raise ValueError(f"Invalid {environment_name}: {values}; unknown={sorted(unknown)}")
    return values

BUDGETS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_BUDGETS", CAMPAIGN_BUDGETS)
ML_SEEDS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_SEEDS", CAMPAIGN_ML_SEEDS)
LOAD_IF_AVAILABLE = os.environ.get("PAPER_SUMMARY_LOAD_IF_AVAILABLE", "1") != "0"

CAMPAIGN = campaign_config(profile=PROFILE)
print(json.dumps({
    "profile": PROFILE,
    "campaign_budgets": CAMPAIGN_BUDGETS,
    "campaign_ml_seeds": CAMPAIGN_ML_SEEDS,
    "budgets_to_run": BUDGETS_TO_RUN,
    "ml_seeds_to_run": ML_SEEDS_TO_RUN,
    "load_if_available": LOAD_IF_AVAILABLE,
    "campaign_signature": campaign_signature(CAMPAIGN),
}, indent=2))


{
  "profile": "PAPER",
  "campaign_budgets": [
    10000,
    100000,
    1000000
  ],
  "campaign_ml_seeds": [
    31082026,
    31082027,
    31082028
  ],
  "budgets_to_run": [
    10000,
    100000,
    1000000
  ],
  "ml_seeds_to_run": [
    31082026,
    31082027,
    31082028
  ],
  "load_if_available": true,
  "campaign_signature": "sha256-e455fa167513"
}


In [4]:
from IPython.display import display

def display_result(result):
    if hasattr(result, "style"):
        display(result.style.format(precision=4).hide(axis="index"))
    elif isinstance(result, dict):
        for name, value in result.items():
            print(f"\n{name}")
            if hasattr(value, "style"):
                display(value.style.format(precision=4).hide(axis="index"))
            else:
                display(value)
    else:
        display(result)


In [5]:
from utils import prepare_slcp_banks

BANK_RESULT = prepare_slcp_banks(
    artifact_root=ARTIFACT_ROOT,
    campaign=CAMPAIGN,
    load_if_available=LOAD_IF_AVAILABLE,
)
display_result(BANK_RESULT)


master: generated 20,000/1,000,000 SLCP pairs
master: generated 40,000/1,000,000 SLCP pairs
master: generated 60,000/1,000,000 SLCP pairs
master: generated 80,000/1,000,000 SLCP pairs
master: generated 100,000/1,000,000 SLCP pairs
master: generated 120,000/1,000,000 SLCP pairs
master: generated 140,000/1,000,000 SLCP pairs
master: generated 160,000/1,000,000 SLCP pairs
master: generated 180,000/1,000,000 SLCP pairs
master: generated 200,000/1,000,000 SLCP pairs
master: generated 220,000/1,000,000 SLCP pairs
master: generated 240,000/1,000,000 SLCP pairs
master: generated 260,000/1,000,000 SLCP pairs
master: generated 280,000/1,000,000 SLCP pairs
master: generated 300,000/1,000,000 SLCP pairs
master: generated 320,000/1,000,000 SLCP pairs
master: generated 340,000/1,000,000 SLCP pairs
master: generated 360,000/1,000,000 SLCP pairs
master: generated 380,000/1,000,000 SLCP pairs
master: generated 400,000/1,000,000 SLCP pairs
master: generated 420,000/1,000,000 SLCP pairs
master: generated

budget,training_rows,validation_rows,master_seed,split_seed,master_fingerprint,split_fingerprint,split_path
10000,8999,1001,31081026,33081026,395f3d1d60363790579d1d5f4b702f38e5443822cf3204a45dfbec1190f6a0c5,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/simulation_banks/slcp_nested_budget_n10000_masterseed31081026.npz
100000,90109,9891,31081026,33081026,395f3d1d60363790579d1d5f4b702f38e5443822cf3204a45dfbec1190f6a0c5,f54e9dbaf307a0a121790dd316984637bc35eea43c37a8adea3686ed2a899e3e,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/simulation_banks/slcp_nested_budget_n100000_masterseed31081026.npz
1000000,900368,99632,31081026,33081026,395f3d1d60363790579d1d5f4b702f38e5443822cf3204a45dfbec1190f6a0c5,ff4fd82d40e02846c75378bf47c41fd4275bdb12e6dea6c84def5dbdec653095,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/simulation_banks/slcp_nested_budget_n1000000_masterseed31081026.npz
